# Generalization test: does the AQI model hold up on data it has never seen?

The final model was trained on 20 Indian cities (CPCB data, 2015-2020). Here it is tested on **Pune**, a city it has never seen, in **2025-26**.

Pune has no NO/NOx, and the full model also uses City and Season features, so this notebook tests a **reduced model**: the same tuned XGBoost + SMOTE, trained on the six pollutants every dataset shares (PM2.5, PM10, NO2, SO2, CO, O3).

**Sections**
1. Data: Pune labels, sanity checks against the Indian data
2. Reduced-feature model (CO/O3 as daily means, the way `city_day.csv` stores them)
3. Where does the error come from? Four ablations
4. Checking how CO/O3 are defined (Delhi)
5. Harmonized model (CO/O3 as 8-hour maxima, matching Pune)
6. Beijing: the untouched confirmation

**How to run:** top to bottom from a fresh kernel. Needs `data/city_day.csv`, `data/city_hour.csv` and `data/pune/city_averaged.csv` (plus the 12 CSVs in `data/beijing/` for section 6). Nothing is read from `data/processed/` or `models/`.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import LabelEncoder

DATA = Path('data')

SHARED = ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3']   # pollutants present in every dataset
ORDER = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
IDX = {c: i for i, c in enumerate(ORDER)}
MAIN3 = ['Good', 'Satisfactory', 'Moderate']            # the only classes with enough Pune days to score
DROP_CITIES = ['Ahmedabad', 'Chennai', 'Gurugram', 'Lucknow', 'Mumbai', 'Patna']  # PM10 structurally missing
AQI_BINS = [-np.inf, 50, 100, 200, 300, 400, np.inf]    # CPCB bands, same as the Indian labels

# Alphabetical class order, identical to the encoder used for the app model
le = LabelEncoder().fit(ORDER)

In [2]:
# ---------- modelling helpers ----------
def fit_xgb(X, y_enc, use_smote=True):
    """Same tuned hyperparameters as the final model. SMOTE is applied to the training data only."""
    if use_smote:
        X, y_enc = SMOTE(random_state=42, k_neighbors=5).fit_resample(X, y_enc)
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=10, learning_rate=0.01,
        subsample=0.6, colsample_bytree=1.0, min_child_weight=1,
        random_state=42, eval_metric='mlogloss',
    )
    model.fit(X, y_enc)
    return model


def evaluate(name, y_true, y_pred):
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(f"Accuracy:   {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1 (macro): {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))


def error_profile(y_true, y_pred, main=MAIN3):
    """Accuracy plus how the errors behave: how far off, and in which direction."""
    t = y_true.map(IDX).values
    p = pd.Series(y_pred).map(IDX).values
    n_wrong = int((t != p).sum())
    return {
        'acc': accuracy_score(y_true, y_pred),
        'within_one': float((np.abs(t - p) <= 1).mean()),
        'f1_main3': f1_score(y_true, y_pred, labels=main, average='macro', zero_division=0),
        'too_high': int((p > t).sum()),      # model said "more polluted" than the label
        'too_low': int((p < t).sum()),
        'over_share': (p > t).sum() / max(n_wrong, 1),
    }


def score_variant(name, model, feats, te_df):
    """One table row: India test score + Pune score for a fitted model."""
    p_in = le.inverse_transform(model.predict(te_df[feats]))
    p_pu = le.inverse_transform(model.predict(pune_ok[feats]))
    m = error_profile(y_pune, p_pu)
    return {
        'variant': name,
        'India_test_acc': round(accuracy_score(te_df['AQI_Bucket'], p_in), 3),
        'India_test_macroF1': round(f1_score(te_df['AQI_Bucket'], p_in, average='macro'), 3),
        'Pune_acc': round(m['acc'], 3),
        'Pune_within_one': round(m['within_one'], 3),
        'Pune_F1_main3': round(m['f1_main3'], 3),
        'Pune_too_high': m['too_high'],
        'Pune_too_low': m['too_low'],
        'Pune_over_share': round(m['over_share'], 3),
    }

In [3]:
# ---------- data helpers ----------
_hourly = None
_eight = None


def load_hourly():
    global _hourly
    if _hourly is None:
        _hourly = pd.read_csv(DATA / 'city_hour.csv', parse_dates=['Datetime'],
                              usecols=['City', 'Datetime', 'CO', 'O3'])
    return _hourly


def eight_hour_max():
    """Per city and day: the maximum of the rolling 8-hour mean of CO and O3 (the CPCB convention)."""
    global _eight
    if _eight is None:
        frames = []
        for city, g in load_hourly().groupby('City'):
            g = g.set_index('Datetime').sort_index()
            out = pd.DataFrame({
                'CO_8h': g['CO'].rolling('8h', min_periods=6).mean().resample('D').max(),
                'O3_8h': g['O3'].rolling('8h', min_periods=6).mean().resample('D').max(),
            })
            out['City'] = city
            frames.append(out.reset_index().rename(columns={'Datetime': 'Date'}))
        _eight = pd.concat(frames, ignore_index=True)
    return _eight


def _split_city(g, test_frac=0.15, val_frac=0.15):
    g = g.sort_values('Date')
    n = len(g)
    a = int(n * (1 - test_frac - val_frac))
    b = int(n * (1 - test_frac))
    return g.iloc[:a], g.iloc[a:b], g.iloc[b:]


def _fill(d, med):
    d = d.copy()
    d[SHARED] = d[SHARED].fillna(med)
    return d


def build_india(co_o3='daily_mean'):
    """Clean the Indian city-day data and split it chronologically *within each city* (70/15/15).

    co_o3: 'daily_mean' = CO/O3 as stored in city_day.csv
           '8h_max'     = CO/O3 rebuilt from city_hour.csv as daily maxima of 8-hour means
    Remaining gaps are filled with TRAIN medians, as in the original pipeline.
    """
    day = raw_day[~raw_day['City'].isin(DROP_CITIES)].copy()
    if co_o3 == '8h_max':
        day = day.merge(eight_hour_max(), on=['City', 'Date'], how='left')
        day['CO'], day['O3'] = day['CO_8h'], day['O3_8h']
    day = day.sort_values(['City', 'Date'])
    day[SHARED] = day.groupby('City')[SHARED].transform(lambda x: x.ffill().bfill())
    day = day.dropna(subset=['AQI', 'AQI_Bucket'])

    parts = [_split_city(g) for _, g in day.groupby('City')]
    tr, va, te = (pd.concat([p[i] for p in parts]) for i in range(3))
    med = tr[SHARED].median()
    return _fill(tr, med), _fill(va, med), _fill(te, med)

## 1. Data

### 1.1 Pune

Pune's AQI is the *maximum* of its per-pollutant sub-indices. On days when only some pollutants reported, the AQI reflects only those, so it is not comparable to a label built from all six. We therefore keep only days where all six pollutants **and** all six sub-indices are present, and do not fill anything: O3, CO and SO2 are about 83% missing, so filling would mean inventing most of the values.

Pune's AQI has decimals while the Indian one is whole numbers, so it is rounded before binning (otherwise 50.4 would land in *Satisfactory* instead of *Good*).

In [4]:
pune_raw = pd.read_csv(DATA / 'pune' / 'city_averaged.csv', parse_dates=['date'])
pune_raw = pune_raw.rename(columns={'pm25': 'PM2.5', 'pm10': 'PM10', 'no2': 'NO2',
                                    'so2': 'SO2', 'co': 'CO', 'o3': 'O3'})
sub_cols = ['sub_pm25', 'sub_pm10', 'sub_no2', 'sub_so2', 'sub_co', 'sub_o3']

pune_ok = pune_raw.dropna(subset=SHARED + sub_cols + ['aqi']).copy()
pune_ok['AQI_Bucket'] = pd.cut(np.floor(pune_ok['aqi'] + 0.5), bins=AQI_BINS, labels=ORDER)
y_pune = pune_ok['AQI_Bucket'].astype(str)

print(f"Pune: {len(pune_raw)} days -> {len(pune_ok)} complete days "
      f"({pune_ok['date'].min().date()} to {pune_ok['date'].max().date()})")
pune_ok['AQI_Bucket'].value_counts().reindex(ORDER)

Pune: 2022 days -> 276 complete days (2025-02-19 to 2026-01-28)


AQI_Bucket
Good             76
Satisfactory     81
Moderate        106
Poor              6
Very Poor         2
Severe            5
Name: count, dtype: int64

### 1.2 Sanity checks against the Indian data

Two things have to line up before any score means anything: the AQI-to-category bands (so labels mean the same thing) and the units (a CO value 1000x off would be a unit problem, not a generalization problem).

In [5]:
raw_day = pd.read_csv(DATA / 'city_day.csv', parse_dates=['Date'])

bands = raw_day.groupby('AQI_Bucket')['AQI'].agg(['min', 'max']).sort_values('min')
scales = pd.DataFrame({'India_median': raw_day[SHARED].median(),
                       'Pune_median': pune_ok[SHARED].median()})
display(bands)
display(scales)

,min,max
AQI_Bucket,,
Good,13.0,50.0
Satisfactory,51.0,100.0
Moderate,101.0,200.0
Poor,201.0,300.0
Very Poor,301.0,400.0
Severe,401.0,2049.0


,India_median,Pune_median
PM2.5,48.57,25.75
PM10,95.68,62.95
NO2,21.69,19.25
SO2,9.16,4.40
CO,0.89,1.25
O3,30.84,43.10


**Result (original run):** the bands match the edges used above (Good <= 50 ... Severe > 400). CO is 0.89 in India and 1.25 in Pune, so both are mg/m3 and there is no unit mismatch. Pune is cleaner on PM2.5, PM10, NO2 and SO2 but higher on O3 and CO. That pattern is explained in section 4.

## 2. Reduced-feature model, CO/O3 as daily means

`build_india('daily_mean')` reproduces the original training data (same cities, same cleaning, same per-city chronological split) for the six shared pollutants only. Training rows should come out as 11,227 / 2,410 / 2,415.

In [6]:
tr_dm, va_dm, te_dm = build_india('daily_mean')
print(len(tr_dm), len(va_dm), len(te_dm), "rows (expect 11227, 2410, 2415)")

model_a = fit_xgb(tr_dm[SHARED], le.transform(tr_dm['AQI_Bucket']))

for name, d in [('India val', va_dm), ('India test', te_dm)]:
    evaluate(f'Reduced model, {name}', d['AQI_Bucket'], le.inverse_transform(model_a.predict(d[SHARED])))

pred_pune_a = le.inverse_transform(model_a.predict(pune_ok[SHARED]))
evaluate(f'Reduced model, Pune ({len(pune_ok)} days)', y_pune, pred_pune_a)
print({k: round(v, 3) for k, v in error_profile(y_pune, pred_pune_a).items()})

pd.DataFrame(confusion_matrix(y_pune, pred_pune_a, labels=ORDER), index=ORDER, columns=ORDER) \
    .rename_axis(index='true', columns='predicted')

11227 2410 2415 rows (expect 11227, 2410, 2415)

Reduced model, India val
Accuracy:   0.8000
F1 (macro): 0.7817
              precision    recall  f1-score   support

        Good       0.71      0.91      0.80       184
    Moderate       0.89      0.75      0.81       956
        Poor       0.61      0.79      0.69       276
Satisfactory       0.83      0.84      0.84       836
      Severe       0.75      0.88      0.81        34
   Very Poor       0.73      0.76      0.74       124

    accuracy                           0.80      2410
   macro avg       0.75      0.82      0.78      2410
weighted avg       0.81      0.80      0.80      2410


Reduced model, India test
Accuracy:   0.8087
F1 (macro): 0.7716
              precision    recall  f1-score   support

        Good       0.70      0.78      0.73       367
    Moderate       0.86      0.81      0.83       714
        Poor       0.60      0.72      0.65        88
Satisfactory       0.85      0.82      0.84      1168
      Sev

predicted,Good,Satisfactory,Moderate,Poor,Very Poor,Severe
true,,,,,,
Good,57,19,0,0,0,0
Satisfactory,1,57,23,0,0,0
Moderate,1,0,79,26,0,0
Poor,0,0,4,2,0,0
Very Poor,0,0,0,1,0,1
Severe,0,0,0,0,0,5


**Result (original run):** Indian test accuracy 0.809, Pune 0.725, with 99.6% of Pune predictions within one category. 69 of Pune's 76 errors are the model calling a day *one step more polluted* than its label. The ordering is right, but the thresholds sit a step high. Poor, Very Poor and Severe have only 6, 2 and 5 Pune days, so their individual scores are noise; use the three-main-class F1.

## 3. Where does the upward shift come from?

Two suspects: **SMOTE** (it can push predictions toward higher classes) and **CO/O3** (they might be measured differently in Pune). Four variants separate them. `Pune_over_share` is the fraction of Pune errors where the model said *more polluted* than the label.

In [7]:
y_tr = le.transform(tr_dm['AQI_Bucket'])
FOUR = ['PM2.5', 'PM10', 'NO2', 'SO2']

rows = [score_variant('A: 6 features + SMOTE (reference)', model_a, SHARED, te_dm)]
for name, feats, smote in [('B: 6 features, no SMOTE', SHARED, False),
                           ('C: no CO/O3 + SMOTE', FOUR, True),
                           ('D: no CO/O3, no SMOTE', FOUR, False)]:
    rows.append(score_variant(name, fit_xgb(tr_dm[feats], y_tr, smote), feats, te_dm))
pd.DataFrame(rows)

,variant,India_test_acc,India_test_macroF1,Pune_acc,Pune_within_one,Pune_F1_main3,Pune_too_high,Pune_too_low,Pune_over_share
0,A: 6 features + SMOTE (reference),0.809,0.772,0.725,0.996,0.772,69,7,0.908
1,"B: 6 features, no SMOTE",0.824,0.777,0.714,0.993,0.710,69,10,0.873
2,C: no CO/O3 + SMOTE,0.733,0.708,0.743,0.938,0.751,20,51,0.282
3,"D: no CO/O3, no SMOTE",0.745,0.708,0.569,0.971,0.470,82,37,0.689


**Result (original run):** removing SMOTE barely changes the over-prediction share (0.91 to 0.87), so SMOTE is not the cause. Removing CO/O3 flips it (0.91 to 0.28) but costs about 7.6 points of Indian accuracy. The shift lives in CO/O3, but dropping them is a diagnostic, not a fix. Variant D is an odd one out and is not a candidate model.

## 4. Checking how CO and O3 are defined

Pune reports the daily maximum of rolling 8-hour averages for CO and O3. The Indian dataset page does not say how its daily CO/O3 were aggregated, so we test it directly: for Delhi, compute both versions from the hourly file and see which one matches `city_day.csv`.

In [8]:
hr_delhi = load_hourly().query("City == 'Delhi'").set_index('Datetime').sort_index()
day_delhi = raw_day[raw_day['City'] == 'Delhi'].set_index('Date')

rows = []
for col in ['CO', 'O3']:
    daily_mean = hr_delhi[col].resample('D').mean()
    daily_max8 = hr_delhi[col].rolling('8h', min_periods=6).mean().resample('D').max()
    cmp = pd.DataFrame({'file': day_delhi[col], 'daily_mean': daily_mean, 'max_8h': daily_max8}).dropna()
    rows.append({'pollutant': col, 'days compared': len(cmp),
                 'median |file - daily mean|': round((cmp['file'] - cmp['daily_mean']).abs().median(), 3),
                 'median |file - 8h max|': round((cmp['file'] - cmp['max_8h']).abs().median(), 3)})
pd.DataFrame(rows)

,pollutant,days compared,median |file - daily mean|,median |file - 8h max|
0,CO,2005,0.002,0.474
1,O3,1924,0.002,19.257


**Result (original run):** the file matches the daily mean almost exactly (median gap 0.002 for both pollutants) and is far from the 8-hour maximum (0.47 for CO, 19 for O3). So the Indian features are daily means while Pune's are 8-hour maxima: the same column names measure different things. O3 peaks in the afternoon, so its 8-hour maximum runs well above its daily mean, which makes Pune look more polluted to a model trained on means. Only Delhi was checked, on the assumption that one pipeline produced every city.

## 5. Harmonized model, CO/O3 as 8-hour maxima

Rebuild CO and O3 in the *training* data as daily maxima of 8-hour means (from `city_hour.csv`), keeping everything else identical, then retrain and re-score. Both versions come from the same function, so the comparison changes exactly one thing.

**Caveat:** Pune was used to diagnose this mismatch, so the Pune score below is a post-diagnosis number, not a blind test. The change followed documented definitions and was a single change with no tuning against Pune. Beijing (section 6) is the untouched check.

In [9]:
tr_h, va_h, te_h = build_india('8h_max')
print(len(tr_h), len(va_h), len(te_h), "rows (should match section 2)")

model_h = fit_xgb(tr_h[SHARED], le.transform(tr_h['AQI_Bucket']))

comparison = pd.DataFrame([
    score_variant('A: daily-mean CO/O3 (as stored)', model_a, SHARED, te_dm),
    score_variant('E: 8-hour-max CO/O3 (harmonized)', model_h, SHARED, te_h),
])
comparison

11227 2410 2415 rows (should match section 2)


,variant,India_test_acc,India_test_macroF1,Pune_acc,Pune_within_one,Pune_F1_main3,Pune_too_high,Pune_too_low,Pune_over_share
0,A: daily-mean CO/O3 (as stored),0.809,0.772,0.725,0.996,0.772,69,7,0.908
1,E: 8-hour-max CO/O3 (harmonized),0.821,0.780,0.826,0.989,0.858,29,19,0.604


In [10]:
pred_pune_h = le.inverse_transform(model_h.predict(pune_ok[SHARED]))
evaluate(f'Harmonized model, Pune ({len(pune_ok)} days)', y_pune, pred_pune_h)

pd.DataFrame(confusion_matrix(y_pune, pred_pune_h, labels=ORDER), index=ORDER, columns=ORDER) \
    .rename_axis(index='true', columns='predicted')


Harmonized model, Pune (276 days)
Accuracy:   0.8261
F1 (macro): 0.5966
              precision    recall  f1-score   support

        Good       0.88      0.92      0.90        76
    Moderate       0.87      0.82      0.84       106
        Poor       0.07      0.17      0.10         6
Satisfactory       0.87      0.80      0.83        81
      Severe       0.83      1.00      0.91         5
   Very Poor       0.00      0.00      0.00         2

    accuracy                           0.83       276
   macro avg       0.59      0.62      0.60       276
weighted avg       0.85      0.83      0.83       276



predicted,Good,Satisfactory,Moderate,Poor,Very Poor,Severe
true,,,,,,
Good,70,6,0,0,0,0
Satisfactory,7,65,9,0,0,0
Moderate,2,4,87,13,0,0
Poor,1,0,4,1,0,0
Very Poor,0,0,0,1,0,1
Severe,0,0,0,0,0,5


**Result (original run):** Pune accuracy 0.725 to 0.833, three-main-class F1 0.772 to 0.868, errors that are "too high" 69 to 28 (and "too low" 7 to 18). Indian test accuracy 0.809 to 0.824. The residual lean (28 high vs 18 low out of 46 errors) is within what chance can produce. The original run dropped 153 rows whose O3 could not be filled (a city with no O3 data); this notebook median-fills them as the original pipeline did, so expect small differences (about a point) from those numbers.

Pune is still only about 276 days (roughly +/-4-5 points of accuracy), and its rare classes have a handful of days each.

## 6. Beijing: the untouched check

Everything above is now **frozen**: `model_h` (trained on India with 8-hour-max CO/O3) is scored on Beijing without any change, so this is the check that was not used for diagnosis.

Beijing has no AQI labels, so we compute them with India's CPCB breakpoints. That makes this a test of whether the model reproduces the AQI formula in a very different environment (different country, climate and pollution sources), **not** a comparison against independent Beijing air-quality categories.

Because the labels are our own calculation, we first check the calculator on Pune, where we have an answer key (Pune publishes its own CPCB sub-indices).

Download the UCI *Beijing Multi-Site Air-Quality Data* (12 station CSVs) into `data/beijing/`.

### 6.1 A CPCB label calculator, validated on Pune

In [11]:
# CPCB concentration breakpoints for the six AQI bands, plus a nominal top end for the Severe band.
# Units: ug/m3, except CO in mg/m3. PM and NO2/SO2 are 24-hour values; CO and O3 are 8-hour maxima.
CPCB_BREAKS = {
    'PM2.5': [0, 30, 60, 90, 120, 250, 380],
    'PM10':  [0, 50, 100, 250, 350, 430, 510],
    'NO2':   [0, 40, 80, 180, 280, 400, 520],
    'SO2':   [0, 40, 80, 380, 800, 1600, 2400],
    'CO':    [0, 1, 2, 10, 17, 34, 50],
    'O3':    [0, 50, 100, 168, 208, 748, 948],
}
AQI_EDGES = [0, 50, 100, 200, 300, 400, 500]
SUB_COLS = {'PM2.5': 'sub_pm25', 'PM10': 'sub_pm10', 'NO2': 'sub_no2',
            'SO2': 'sub_so2', 'CO': 'sub_co', 'O3': 'sub_o3'}


def cpcb_aqi(df):
    """AQI = the largest per-pollutant sub-index (piecewise linear inside each band, capped at 500)."""
    subs = pd.DataFrame({p: np.interp(df[p].to_numpy(float), CPCB_BREAKS[p], AQI_EDGES) for p in SHARED},
                        index=df.index)
    return subs.max(axis=1), subs


def to_bucket(aqi):
    return pd.cut(np.floor(aqi + 0.5), bins=AQI_BINS, labels=ORDER)


# Validation: recompute Pune's sub-indices and categories from its concentrations
mine_aqi, mine_sub = cpcb_aqi(pune_ok)
sub_gap = pd.Series({p: (mine_sub[p] - pune_ok[SUB_COLS[p]]).abs().median() for p in SHARED}).round(2)
agree = (to_bucket(mine_aqi).astype(str) == y_pune).mean()

print("Median |our sub-index - Pune's sub-index| per pollutant:")
print(sub_gap.to_string())
print(f"\nCategory agreement with Pune's own labels: {agree:.1%} ({len(pune_ok)} days)")

Median |our sub-index - Pune's sub-index| per pollutant:
PM2.5    0.07
PM10     0.00
NO2      0.05
SO2      0.03
CO       1.20
O3       0.00

Category agreement with Pune's own labels: 100.0% (276 days)


Small sub-index gaps (a few points at most) and high category agreement mean the calculator is trustworthy enough to label Beijing. A large gap for one pollutant would point at a wrong breakpoint or unit for that pollutant, to be fixed before going further. Small differences are expected, especially for CO, whose bands are narrow: CPCB's published bands have 1-unit gaps (e.g. 30 to 31 for PM2.5) that this calculator interpolates across.

### 6.2 Build Beijing's daily city series

Same conventions as the harmonized Indian training data: stations averaged into one city series; PM2.5, PM10, NO2, SO2 as daily means; CO and O3 as the daily maximum of 8-hour means; CO converted from ug/m3 to mg/m3. A day needs at least 16 valid hours per pollutant (the CPCB rule), and days with any pollutant missing are dropped rather than filled.

In [12]:
bj_files = sorted((DATA / 'beijing').glob('*.csv'))
print(len(bj_files), "CSV files found (the UCI download has 12, one per station):")
frames = []
for f in bj_files:
    d = pd.read_csv(f)
    print(f"  {f.name}: {len(d)} rows")
    frames.append(d)
bj_raw = pd.concat(frames, ignore_index=True)

need = {'year', 'month', 'day', 'hour', 'station'} | set(SHARED)
missing = need - set(bj_raw.columns)
assert not missing, f"Beijing files lack columns {sorted(missing)}. Columns found: {list(bj_raw.columns)}"

bj_raw['datetime'] = pd.to_datetime(bj_raw[['year', 'month', 'day', 'hour']])
n_before = len(bj_raw)
bj_raw = bj_raw.drop_duplicates(subset=['station', 'datetime'])   # a repeated file would double-weight a station
print(f"Dropped {n_before - len(bj_raw)} duplicate station-hours (repeated files)")
per_station = bj_raw.groupby('station').size()
print(f"Rows per station after de-duplication: min {per_station.min()}, max {per_station.max()} (expect 35064 each)")
print(bj_raw['station'].nunique(), "stations,", bj_raw['datetime'].min(), "to", bj_raw['datetime'].max())
print("\n% missing per pollutant (hourly, all stations):")
print((bj_raw[SHARED].isnull().mean() * 100).round(1).to_string())

14 CSV files found (the UCI download has 12, one per station):
  data.csv: 503 rows
  PRSA_Data_Aotizhongxin_20130301-20170228.csv: 35064 rows
  PRSA_Data_Changping_20130301-20170228.csv: 35064 rows
  PRSA_Data_Dingling_20130301-20170228.csv: 35064 rows
  PRSA_Data_Dongsi_20130301-20170228.csv: 35064 rows
  PRSA_Data_Guanyuan_20130301-20170228.csv: 35064 rows
  PRSA_Data_Gucheng_20130301-20170228.csv: 35064 rows
  PRSA_Data_Huairou_20130301-20170228.csv: 35064 rows
  PRSA_Data_Nongzhanguan_20130301-20170228.csv: 35064 rows
  PRSA_Data_Shunyi_20130301-20170228.csv: 35064 rows
  PRSA_Data_Tiantan_20130301-20170228.csv: 35064 rows
  PRSA_Data_Wanliu_20130301-20170228.csv: 35064 rows
  PRSA_Data_Wanshouxigong_20130301-20170228.csv: 35064 rows
  test.csv: 503 rows
Dropped 1005 duplicate station-hours (repeated files)
Rows per station after de-duplication: min 35064, max 35064 (expect 35064 each)
12 stations, 2013-03-01 00:00:00 to 2017-02-28 23:00:00

% missing per pollutant (hourly, all st

In [13]:
MIN_HOURS = 16

bj_hourly = (bj_raw.assign(CO=bj_raw['CO'] / 1000)       # ug/m3 -> mg/m3, as in the Indian data
             .groupby('datetime')[SHARED].mean()          # stations -> one city series
             .asfreq('h'))                                 # regular hourly index so time-based rolling works
valid_hours = bj_hourly.resample('D').count()

daily = pd.DataFrame(index=valid_hours.index)
for col in ['PM2.5', 'PM10', 'NO2', 'SO2']:
    daily[col] = bj_hourly[col].resample('D').mean()
for col in ['CO', 'O3']:
    daily[col] = bj_hourly[col].rolling('8h', min_periods=6).mean().resample('D').max()
daily = daily.where(valid_hours[SHARED] >= MIN_HOURS)

bj_ok = daily.dropna().copy()
bj_aqi, _ = cpcb_aqi(bj_ok)
bj_ok['AQI_Bucket'] = to_bucket(bj_aqi)
y_bj = bj_ok['AQI_Bucket'].astype(str)

print(f"Beijing: {len(daily)} days -> {len(bj_ok)} complete days "
      f"({bj_ok.index.min().date()} to {bj_ok.index.max().date()})")
assert 0.05 < bj_ok['CO'].median() < 10, "Beijing CO does not look like mg/m3 - check the unit conversion"

display(bj_ok['AQI_Bucket'].value_counts().reindex(ORDER))
display(pd.DataFrame({'India_median': raw_day[SHARED].median(), 'Beijing_median': bj_ok[SHARED].median()}))

Beijing: 1461 days -> 1453 complete days (2013-03-01 to 2017-02-28)


AQI_Bucket
Good             14
Satisfactory    369
Moderate        447
Poor            246
Very Poor       332
Severe           45
Name: count, dtype: int64

,India_median,Beijing_median
PM2.5,48.57,60.857639
PM10,95.68,89.404356
NO2,21.69,44.400884
SO2,9.16,9.317551
CO,0.89,1.321875
O3,30.84,84.062500


### 6.3 Score the frozen model

In [14]:
def score_external(name, model, df, y_true, main):
    pred = le.inverse_transform(model.predict(df[SHARED]))
    m = error_profile(y_true, pred, main=main)
    return {
        'model': name,
        'acc': round(m['acc'], 3),
        'within_one': round(m['within_one'], 3),
        f'F1_main({len(main)})': round(m['f1_main3'], 3),
        'too_high': m['too_high'],
        'too_low': m['too_low'],
        'over_share': round(m['over_share'], 3),
    }


# Score only the classes with at least 30 Beijing days; smaller classes are noise
big = [c for c in ORDER if (y_bj == c).sum() >= 30]
print("Classes scored in the F1 column:", big)

pd.DataFrame([
    score_external('E: harmonized model (frozen)', model_h, bj_ok, y_bj, big),
    score_external('A: daily-mean model, same Beijing features', model_a, bj_ok, y_bj, big),
])

Classes scored in the F1 column: ['Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']


,model,acc,within_one,F1_main(5),too_high,too_low,over_share
0,E: harmonized model (frozen),0.714,0.959,0.728,60,355,0.145
1,"A: daily-mean model, same Beijing features",0.641,0.975,0.686,207,314,0.397


In [15]:
pred_bj = le.inverse_transform(model_h.predict(bj_ok[SHARED]))
evaluate(f'Harmonized model, Beijing ({len(bj_ok)} days)', y_bj, pred_bj)

pd.DataFrame(confusion_matrix(y_bj, pred_bj, labels=ORDER), index=ORDER, columns=ORDER) \
    .rename_axis(index='true', columns='predicted')


Harmonized model, Beijing (1453 days)
Accuracy:   0.7144
F1 (macro): 0.7088
              precision    recall  f1-score   support

        Good       0.67      0.57      0.62        14
    Moderate       0.73      0.64      0.68       447
        Poor       0.54      0.63      0.58       246
Satisfactory       0.70      0.97      0.81       369
      Severe       0.94      0.76      0.84        45
   Very Poor       0.92      0.59      0.72       332

    accuracy                           0.71      1453
   macro avg       0.75      0.69      0.71      1453
weighted avg       0.74      0.71      0.71      1453



predicted,Good,Satisfactory,Moderate,Poor,Very Poor,Severe
true,,,,,,
Good,8,6,0,0,0,0
Satisfactory,0,357,12,0,0,0
Moderate,4,121,287,35,0,0
Poor,0,21,64,156,5,0
Very Poor,0,6,29,99,196,2
Severe,0,0,0,0,11,34


**How to read it.** `E` is the headline: the frozen harmonized model on a city and country it has never seen, scored on all classes with at least 30 days. `A` is the replication of the Pune diagnosis on data that was not used for it: it gets the same Beijing features (8-hour-max CO/O3), but was trained on daily-mean CO/O3. If `A` over-predicts severity (high `over_share`, lower accuracy) and `E` does not, the CO/O3 explanation holds up on a second, independent dataset.

**Caveats to state honestly:** the labels are our own CPCB-style calculation, not Beijing's official categories; the 12 stations are averaged into one series; and a model trained only on India is being asked about a different pollution regime (Beijing has far more Poor/Very Poor/Severe days than the Indian cities).

### 6.4 Two follow-up questions

**(a) Can we trust the labels in the upper bands?** The Pune check covered mostly Good to Moderate days (only about a dozen Poor/Very Poor/Severe). The Indian data has thousands of upper-band days with the dataset's own labels, so we compare our calculator against them. The dataset's AQI can also be raised by pollutants we do not compute (e.g. NH3), so expect our label to be equal or *lower*, never systematically higher. The comparison is run with both CO/O3 conventions, which also tests which one the dataset's labels follow.

**(b) Why does the model under-call severity on Beijing?** One hypothesis: Beijing's PM2.5 is a larger share of its PM10 than India's, and the model leans heavily on PM10, so for the same PM2.5 it sees a milder picture. The table groups Beijing days by the pollutant that actually sets the label and shows how far off the model is on each.

In [16]:
# (a) our calculator vs the Indian dataset's own labels
all_dm = pd.concat([tr_dm, va_dm, te_dm])
all_h = pd.concat([tr_h, va_h, te_h])
for name, d in [('daily-mean CO/O3', all_dm), ('8-hour-max CO/O3', all_h)]:
    ours = to_bucket(cpcb_aqi(d)[0]).astype(str)
    print(f"{name}: our labels match the dataset's on {(ours == d['AQI_Bucket']).mean():.1%} of {len(d)} Indian days")

ours_h = to_bucket(cpcb_aqi(all_h)[0]).astype(str)
display(pd.crosstab(all_h['AQI_Bucket'], ours_h).reindex(index=ORDER, columns=ORDER).fillna(0).astype(int)
        .rename_axis(index="dataset's label", columns='our label'))

daily-mean CO/O3: our labels match the dataset's on 71.3% of 16052 Indian days
8-hour-max CO/O3: our labels match the dataset's on 75.6% of 16052 Indian days


our label,Good,Satisfactory,Moderate,Poor,Very Poor,Severe
dataset's label,,,,,,
Good,996,196,60,2,12,0
Satisfactory,565,4703,698,16,69,0
Moderate,9,885,4696,201,206,6
Poor,1,12,392,795,295,5
Very Poor,0,1,9,139,727,41
Severe,0,0,0,6,96,213


In [17]:
# (b) PM10 vs PM2.5, feature importance, and where the model falls short on Beijing
def pm_ratio(d):
    return (d['PM10'] / d['PM2.5']).median()

print(f"Median PM10 / PM2.5   India: {pm_ratio(tr_h):.2f}   Pune: {pm_ratio(pune_ok):.2f}   Beijing: {pm_ratio(bj_ok):.2f}")
display(pd.Series(model_h.feature_importances_, index=SHARED).sort_values(ascending=False).round(3).rename('importance'))

_, bj_subs = cpcb_aqi(bj_ok)
res = pd.DataFrame({'sets_the_label': bj_subs.idxmax(axis=1), 'true': y_bj, 'pred': pred_bj}, index=bj_ok.index)
res['step_error'] = res['pred'].map(IDX) - res['true'].map(IDX)      # negative = model said less polluted
res.groupby('sets_the_label')['step_error'].agg(
    days='size', mean_step_error='mean', share_under_called=lambda g: (g < 0).mean()).round(2)

Median PM10 / PM2.5   India: 2.09   Pune: 2.48   Beijing: 1.37


PM2.5    0.409
PM10     0.325
CO       0.109
O3       0.059
NO2      0.051
SO2      0.046
Name: importance, dtype: float32

,days,mean_step_error,share_under_called
sets_the_label,,,
CO,132,-0.07,0.09
NO2,22,0.05,0.00
O3,639,-0.50,0.42
PM10,62,0.03,0.10
PM2.5,598,-0.06,0.12


### 6.5 A cleaner label check

The 6.4(a) comparison used the pipeline's *filled* values (gaps carried over from neighbouring days) and ignored NH3, while the dataset's own AQI uses only what was measured that day and can include NH3. So here we compare only on days where all six pollutants were genuinely measured, with and without NH3, and with both CO/O3 conventions. If agreement is high, the calculator reproduces the dataset's labels and Beijing's labels are comparable to the training labels. If not, the training labels follow a convention we have not matched yet, and part of Beijing's error may be a labelling gap rather than a generalization gap.

### 6.6 Is the under-calling an O3 coverage problem?

The 6.4(b) table showed the under-calling concentrated on days where O3 sets the label, while PM2.5-driven days are predicted well. Tree models cannot predict beyond the range of values they saw in training, so the question is how many training days had O3 driving a Poor-or-worse label. If almost none, the model has never seen that regime.

In [20]:
print(len(bj_files), "files found:")
for f in bj_files:
    print(" ", f.name)

n_before = len(bj_raw)
bj_raw = bj_raw.drop_duplicates(subset=['station', 'datetime'])
print("duplicate station-hours dropped:", n_before - len(bj_raw))
print(bj_raw.groupby('station').size().agg(['min', 'max']).to_string())   # expect 35064 each

14 files found:
  data.csv
  PRSA_Data_Aotizhongxin_20130301-20170228.csv
  PRSA_Data_Changping_20130301-20170228.csv
  PRSA_Data_Dingling_20130301-20170228.csv
  PRSA_Data_Dongsi_20130301-20170228.csv
  PRSA_Data_Guanyuan_20130301-20170228.csv
  PRSA_Data_Gucheng_20130301-20170228.csv
  PRSA_Data_Huairou_20130301-20170228.csv
  PRSA_Data_Nongzhanguan_20130301-20170228.csv
  PRSA_Data_Shunyi_20130301-20170228.csv
  PRSA_Data_Tiantan_20130301-20170228.csv
  PRSA_Data_Wanliu_20130301-20170228.csv
  PRSA_Data_Wanshouxigong_20130301-20170228.csv
  test.csv
duplicate station-hours dropped: 0
min    35064
max    35064


In [21]:
NH3_BREAKS = [0, 200, 400, 800, 1200, 1800, 2400]

raw_ok = raw_day[~raw_day['City'].isin(DROP_CITIES)].merge(eight_hour_max(), on=['City', 'Date'], how='left')

rows = []
for co_o3, needed in [('daily_mean', SHARED), ('8h_max', ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO_8h', 'O3_8h'])]:
    d = raw_ok.dropna(subset=needed + ['AQI_Bucket']).copy()
    if co_o3 == '8h_max':
        d['CO'], d['O3'] = d['CO_8h'], d['O3_8h']
    base, _ = cpcb_aqi(d)
    nh3 = pd.Series(np.interp(d['NH3'].to_numpy(float), NH3_BREAKS, AQI_EDGES), index=d.index)
    for label, aqi in [('six pollutants', base), ('+ NH3 where measured', np.fmax(base, nh3))]:
        ours = to_bucket(aqi).astype(str)
        rows.append({'CO/O3 as': co_o3, 'calculator': label, 'days': len(d),
                     'match': round((ours == d['AQI_Bucket']).mean(), 3),
                     'ours_higher': int((ours.map(IDX) > d['AQI_Bucket'].map(IDX)).sum()),
                     'ours_lower': int((ours.map(IDX) < d['AQI_Bucket'].map(IDX)).sum())})
pd.DataFrame(rows)

,CO/O3 as,calculator,days,match,ours_higher,ours_lower
0,daily_mean,six pollutants,13758,0.737,951,2662
1,daily_mean,+ NH3 where measured,13758,0.737,951,2662
2,8h_max,six pollutants,13689,0.784,1106,1849
3,8h_max,+ NH3 where measured,13689,0.784,1106,1849


In [22]:
_, tr_subs = cpcb_aqi(tr_h)
tr_dom = tr_subs.idxmax(axis=1)
bj_dom = bj_subs.idxmax(axis=1)

n_tr = int(((tr_dom == 'O3') & (tr_subs['O3'] > 200)).sum())
n_bj = int(((bj_dom == 'O3') & (bj_subs['O3'] > 200)).sum())
print(f"O3 sets the label AND AQI > 200:  India training {n_tr} of {len(tr_h)} days   |   Beijing {n_bj} of {len(bj_ok)} days")

display(pd.DataFrame({'India_train_O3': tr_h['O3'].describe(percentiles=[.5, .9, .99]),
                      'Beijing_O3': bj_ok['O3'].describe(percentiles=[.5, .9, .99])}).round(1))

res['correct'] = res['pred'] == res['true']
res.groupby('sets_the_label')['correct'].agg(days='size', accuracy='mean').round(3)

O3 sets the label AND AQI > 200:  India training 14 of 11227 days   |   Beijing 193 of 1453 days


,India_train_O3,Beijing_O3
count,11227.0,1453.0
mean,50.7,98.2
std,33.6,63.4
min,0.1,2.5
50%,43.4,84.1
90%,95.9,198.1
99%,149.2,246.3
max,369.1,310.6


,days,accuracy
sets_the_label,,
CO,132,0.864
NO2,22,0.955
O3,639,0.563
PM10,62,0.774
PM2.5,598,0.828


In [23]:
# 1) what are the two extra files?
for name in ['data.csv', 'test.csv']:
    d = pd.read_csv(DATA / 'beijing' / name, nrows=5)
    print(name, '->', d.columns.tolist())

# 2) rebuild Beijing from the 12 station files only, then re-score
bj_raw = pd.concat([pd.read_csv(f) for f in sorted((DATA / 'beijing').glob('PRSA_Data_*.csv'))], ignore_index=True)
bj_raw['datetime'] = pd.to_datetime(bj_raw[['year', 'month', 'day', 'hour']])
print(len(bj_raw), "rows,", bj_raw['station'].nunique(), "stations (expect 420768 rows, 12 stations)")

bj_hourly = bj_raw.assign(CO=bj_raw['CO'] / 1000).groupby('datetime')[SHARED].mean().asfreq('h')
valid_hours = bj_hourly.resample('D').count()
daily = pd.DataFrame(index=valid_hours.index)
for col in ['PM2.5', 'PM10', 'NO2', 'SO2']:
    daily[col] = bj_hourly[col].resample('D').mean()
for col in ['CO', 'O3']:
    daily[col] = bj_hourly[col].rolling('8h', min_periods=6).mean().resample('D').max()
daily = daily.where(valid_hours[SHARED] >= 16)

bj_ok = daily.dropna().copy()
bj_aqi, bj_subs = cpcb_aqi(bj_ok)
bj_ok['AQI_Bucket'] = to_bucket(bj_aqi)
y_bj = bj_ok['AQI_Bucket'].astype(str)
print(len(bj_ok), "complete days (before: 1453)")

big = [c for c in ORDER if (y_bj == c).sum() >= 30]
pd.DataFrame([score_external('E: harmonized (frozen)', model_h, bj_ok, y_bj, big),
              score_external('A: daily-mean model', model_a, bj_ok, y_bj, big)])

data.csv -> ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
test.csv -> ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
420768 rows, 12 stations (expect 420768 rows, 12 stations)
1453 complete days (before: 1453)


,model,acc,within_one,F1_main(5),too_high,too_low,over_share
0,E: harmonized (frozen),0.714,0.959,0.728,60,355,0.145
1,A: daily-mean model,0.641,0.975,0.686,207,314,0.397


In [24]:
a = score_variant('A: daily-mean CO/O3', model_a, SHARED, te_dm)
e = score_variant('E: 8-hour-max CO/O3', model_h, SHARED, te_h)

rows = []
for r in (a, e):
    rows += [
        {'dataset': 'India test (20 cities)', 'model': r['variant'], 'days': len(te_dm), 'accuracy': r['India_test_acc']},
        {'dataset': 'Pune (unseen city)',     'model': r['variant'], 'days': len(pune_ok), 'accuracy': r['Pune_acc']},
    ]

pred_bj = {}
for name, m in [('A: daily-mean CO/O3', model_a), ('E: 8-hour-max CO/O3', model_h)]:
    pred_bj[name] = le.inverse_transform(m.predict(bj_ok[SHARED]))
    rows.append({'dataset': 'Beijing (unseen country)', 'model': name, 'days': len(bj_ok),
                 'accuracy': round((pred_bj[name] == y_bj.values).mean(), 3)})

# Beijing split by which pollutant sets the label, for the frozen model E
o3_day = (bj_subs.idxmax(axis=1) == 'O3').values
p = pred_bj['E: 8-hour-max CO/O3']
for label, mask in [('Beijing, days NOT set by O3', ~o3_day), ('Beijing, days set by O3', o3_day)]:
    rows.append({'dataset': label, 'model': 'E: 8-hour-max CO/O3', 'days': int(mask.sum()),
                 'accuracy': round((p[mask] == y_bj.values[mask]).mean(), 3)})

pd.DataFrame(rows)

,dataset,model,days,accuracy
0,India test (20 cities),A: daily-mean CO/O3,2415,0.809
1,Pune (unseen city),A: daily-mean CO/O3,276,0.725
2,India test (20 cities),E: 8-hour-max CO/O3,2415,0.821
3,Pune (unseen city),E: 8-hour-max CO/O3,276,0.826
4,Beijing (unseen country),A: daily-mean CO/O3,1453,0.641
5,Beijing (unseen country),E: 8-hour-max CO/O3,1453,0.714
6,"Beijing, days NOT set by O3",E: 8-hour-max CO/O3,814,0.833
7,"Beijing, days set by O3",E: 8-hour-max CO/O3,639,0.563


In [25]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

P = 'data/processed/'
leak = ['Benzene', 'Toluene', 'Xylene']
X_train = pd.read_csv(P + 'X_train.csv').drop(columns=leak, errors='ignore')
X_val   = pd.read_csv(P + 'X_val.csv').drop(columns=leak, errors='ignore')
y_train = pd.read_csv(P + 'y_train.csv').squeeze('columns')
y_val   = pd.read_csv(P + 'y_val.csv').squeeze('columns')
print(X_train.shape, "(expect 11227, 31)")

med = X_train.median()
X_train, X_val = X_train.fillna(med), X_val.fillna(med)
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_val = scaler.transform(X_train), scaler.transform(X_val)

models = {'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
          'Decision Tree (no depth limit)': (DecisionTreeClassifier(random_state=42), False),
          'KNN (k=5)': (KNeighborsClassifier(n_neighbors=5), True)}
rows, preds = [], {}
for name, (m, scaled) in models.items():
    Xa, Xb = (Xs_train, Xs_val) if scaled else (X_train, X_val)
    m.fit(Xa, y_train); preds[name] = m.predict(Xb)
    rows.append({'model': name, 'acc': round(accuracy_score(y_val, preds[name]), 3),
                 'f1_macro': round(f1_score(y_val, preds[name], average='macro'), 3),
                 'f1_weighted': round(f1_score(y_val, preds[name], average='weighted'), 3)})

labels = ['Good', 'Satisfactory', 'Moderate', 'Poor', 'Very Poor', 'Severe']
cm = pd.DataFrame(confusion_matrix(y_val, preds['Logistic Regression'], labels=labels), index=labels, columns=labels)
print("LR: Good called Satisfactory:", cm.loc['Good', 'Satisfactory'], "of", cm.loc['Good'].sum())
print("LR: Severe recall:", round(cm.loc['Severe', 'Severe'] / cm.loc['Severe'].sum(), 2))
pd.DataFrame(rows)

(11227, 31) (expect 11227, 31)
LR: Good called Satisfactory: 127 of 184
LR: Severe recall: 0.76


,model,acc,f1_macro,f1_weighted
0,Logistic Regression,0.749,0.692,0.740
1,Decision Tree (no depth limit),0.730,0.668,0.729
2,KNN (k=5),0.681,0.646,0.678
